In [1]:
from pyspark.sql import SparkSession

# Initialize Interactive Spark Session with limited resources
spark = SparkSession.builder \
    .appName("JupyterInteractive") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.cores", "2") \
    .config("spark.cores.max", "2") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.jars.ivy", "/tmp/.ivy2") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark Session Ready with 2 Cores!")


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/.ivy2/cache
The jars for the packages stored in: /tmp/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-20c69619-6390-455d-b74b-f04e2091b182;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 332ms :: artifacts dl 11ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 fr

✅ Spark Session Ready with 2 Cores!


In [2]:
df = spark.read.format("delta").load("s3a://delta-lake/bronze/hotels")
total_hotels = df.count()
print(f"Total Hotels in Bronze Table: {total_hotels}\n")
df.createOrReplaceTempView("hotels_view")

26/03/09 04:02:01 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/09 04:02:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:==================================================>      (44 + 2) / 50]

Total Hotels in Bronze Table: 2584



In [3]:
df = spark.read.format("delta").load("s3a://delta-lake/bronze/hotel_pairs")
df.createOrReplaceTempView("pairs_view")

In [4]:
df = spark.read.format("delta").load("s3a://delta-lake/bronze/final_clusters")
df.createOrReplaceTempView("clusters_view")
df = spark.read.format("delta").load("s3a://delta-lake/bronze/comparison_audit_log")
df.createOrReplaceTempView("comparison_audit_log")

In [5]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

spark.sql("""
    SELECT 
        count(distinct cluster_id)
        --*
    FROM clusters_view
    limit 3
""").toPandas()

,count(DISTINCT cluster_id)
0,2077


In [6]:
spark.sql("""
    SELECT 
        count(*)
    FROM hotels_view
    WHERE providerId = 'bookingcom'
    --limit 5
""").toPandas()

,count(1)
0,1432


In [7]:
spark.sql("""
SELECT
  COUNT(*) AS audit_true,
  SUM(CASE WHEN size(failed_conditions) = 0 THEN 1 ELSE 0 END) AS audit_true_no_failures
FROM comparison_audit_log
WHERE is_matched = true;
""").toPandas()

,audit_true,audit_true_no_failures
0,519,519


In [8]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

spark.sql("""
SELECT
    *
FROM comparison_audit_log
WHERE is_matched = false
AND score >= 0.6
ORDER BY score DESC
LIMIT 100
""").toPandas()

,hotel_id,hotel_name,compared_with_id,compared_with_name,score,status,is_matched,failed_conditions,comparison_at,geo_distance_km,name_score_jaccard,normalized_name_score_jaccard,name_score_lcs,normalized_name_score_lcs,name_score_levenshtein,normalized_name_score_levenshtein,name_score_sbert,normalized_name_score_sbert,address_line1_score,address_sbert_score,star_ratings_score,postal_code_match,phone_match_score,email_match_score,fax_match_score
0,749d6fc5cca9d34e65d537828d7e12d1310354b5868af30eeddec205037a33ae,hotel cliffton,ea767d7f7a4c2520184427bb77b229a141f002ada0554138c3f7df781930bb45,hotel cliffton,0.947757,MATCHED,False,"[Failed OR Block (postal_code_match, geo_distance_km)]",2026-03-09 04:00:32.031571,0.112437,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.79,0.687567,0.8,0.0,0.5,0.5,0.5
1,1ef6c5efce3281df12f91de894af55704eb6e5021e52624ebcb521ce1be83c50,hotel signature inn,773bcd587b4c2ceac106a7a51713e132abb47ad63f2b9f3d0b785cdc3275e2b6,hotel signature inn,0.877167,MATCHED,False,"[Failed OR Block (postal_code_match, geo_distance_km)]",2026-03-09 04:00:32.031571,0.218598,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.27,0.501673,0.8,0.0,0.5,0.5,0.5
2,9b390f93b34d6ce09bdde0b5fdd1f4b5ba9baf95eb0726b876c7dc406e33743d,"treebo premium heritage grand, thane",6c303df8e7c2389572252873dffbc775a5db01f51dc13018b71bd4637e45b224,treebo premium heritage grand -majiwada thane,0.822339,MANUAL_REVIEW,False,"[Failed OR Block (name_score_jaccard, name_score_lcs...)]",2026-03-09 04:00:32.031571,0.036032,0.833333,0.800000,0.681818,0.763158,0.890000,0.870000,0.890972,0.862503,0.76,0.871605,0.8,1.0,0.5,0.5,0.5
3,0a936c9b3c1401da486969d8aa25183addf9767c563e4bdb25265f89b40cc686,fabexpress silver lake,04e89bda6dd87e5e9d07b7e1869d565dbaa8eeb5ba613b952b3dfa29ff0efa2b,fabexpress silver lake - nr bkc,0.816234,MANUAL_REVIEW,False,"[Failed OR Block (name_score_jaccard, name_score_lcs...)]",2026-03-09 04:00:32.031571,0.018411,0.600000,0.600000,0.758621,0.758621,0.860000,0.860000,0.862549,0.862549,1.00,1.000000,1.0,1.0,0.5,0.5,0.5
4,2feab9c826960b28c93227b85624b3c5b9774b6b5768ef645176ea430f8d93de,hotel airport international,582c20b447792998eb409ad6ed796e7f77834b3c7b55be5c7a7bfe49c25b6fbd,hotel airport international mumbai,0.797444,UNMATCHED,False,"[Failed OR Block (name_score_jaccard, name_score_lcs...)]",2026-03-09 04:00:32.031571,0.008253,0.750000,0.750000,0.794118,0.794118,0.890000,0.890000,0.780767,0.780767,0.73,0.814673,1.0,1.0,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,28e3f041f90592569173f2175d5d97173d4f34083275a8b6f9405ddb3f5d9575,hotel bkc prime,31523a5a51389d84a202a595f33dc2be80cb0b50781e041cfbe1f5ce164931de,hotel bkc mannat,0.649057,UNMATCHED,False,"[Failed OR Block (name_score_jaccard, name_score_lcs...)]",2026-03-09 03:50:12.265160,0.034394,0.500000,0.500000,0.625000,0.625000,0.710000,0.710000,0.746655,0.746655,0.61,0.717257,0.8,1.0,0.0,0.5,0.5
96,5a06d694aee8fd3dab522e24482fcd330854b5c89dede0c29513e4a1ee8adbae,zara dormitory,667faa7449ae5873d6b571838dafe9cb56ca1b25a6c727f87fc2a2f7f91e62e2,asra dormitory,0.649050,UNMATCHED,False,"[Failed OR Block (postal_code_match, geo_distance_km)]",2026-03-09 03:50:12.265160,0.333690,0.333333,0.333333,0.857143,0.857143,0.930000,0.930000,0.692766,0.692766,0.38,0.484016,1.0,0.0,0.0,0.5,0.5
97,05757b0cafd36b3e02e08350bc003402cabdec85f5f25ee94710637929c55da6,"angel views 602, chapel road, bandra west by connekt homes",c6c0d4289766a0a5c490ffe21ad13c91883a0f865f9d16a4f881939bef42ede6,"saba 301, subko coffee, chapel road, bandra west by connekt homes",0.646993,UNMATCHED,False,"[Failed OR Block (name_score_jaccard, name_score_lcs...)]",2026-03-09 04:00:32.031571,0.197676,0.500000,0.461538,0.661290,0.644068,0.760000,0.750000,0.645783,0.593492,0.70,0.753753,1.0,1.0,0.5,0.5,0.5
98,4b253e8e3f53f60a43aeacd6f71f98566af318d82de662c2adee4acad39527ee,fabhotel seven hills - nr mumbai internatio

In [11]:
spark.sql("""
select * 
from pairs_view
where providerHotelId_i = '100843987'
and providerHotelId_j = '7206455'
""").toPandas()

,id_i,id_j,providerHotelId_i,providerHotelId_j,name_i,name_j,uid_i,uid_j,normalized_name_i,normalized_name_j,combined_address_i,combined_address_j,geo_distance_km,name_score_jaccard,normalized_name_score_jaccard,name_score_lcs,normalized_name_score_lcs,name_score_levenshtein,normalized_name_score_levenshtein,name_score_sbert,normalized_name_score_sbert,star_ratings_score,address_line1_score,postal_code_match,country_match,address_sbert_score,phone_match_score,email_match_score,fax_match_score,property_type_score,name_unit_score,address_unit_score,supplier_score,providerName_i,providerName_j,type_i,type_j,name_score_containment,normalized_name_score_containment,average_name_score,average_normalized_name_score
0,52195034,70683244,100843987,7206455,hotel twigo inn,fabhotel twigo inn,5aab44f9b6c68d97cb9c0f3669e08a76249f19e8043e76a0085f0221134b06bd,2c70930fda9aeda84eb77ec0a237b593c580ba4d6cc9881db0432ff4870977bf,hotel twigo inn,fabhotel twigo inn,None,None,0.011321,0.5,0.5,0.833333,0.833333,0.91,0.91,0.791423,0.791423,0.2,0.65,0.0,1.0,0.813842,0.5,0.5,0.5,1.0,1.0,0.0,0,ean,bookingcom,hotel,hotel,0.666667,0.666667,0.758689,0.758689
